# 提升 AUC 分數的步驟說明

> 對應檔案：`Give_Me_Some_Credit.ipynb`　|　目前分數：**AUC-ROC = 0.5825**（幾乎等同隨機猜測 0.5）

本 Notebook **不改動原始程式碼**，只說明要把 AUC 提高應該**新增**哪些步驟，並附上可執行的範例。
步驟依「投資報酬率（影響大小／實作難度）」由高到低排列，建議由上往下做。

| 步驟 | 重點 | 對 AUC 的影響 |
|------|------|----------------|
| 0 | `predict_proba` 取代 `predict` | ★★★★★（最關鍵，~0.58 → ~0.85） |
| 1 | `scoring="roc_auc"` | ★★★ |
| 2 | 合理超參數範圍 | ★★★ |
| 3 | 缺失值填補 | ★★ |
| 4 | 離群值／異常碼清理 | ★★ |
| 5 | 類別不平衡 class_weight | ★★ |
| 6 | 特徵工程 | ★★ |
| 7 | 梯度提升模型 | ★★★★（通常勝過 RF） |
| 8 | 交叉驗證 AUC 評估 | 評估更穩健 |
| 9 | 提交檔 Id 對齊 | 正確提交 |

> 競賽歷史上，本題前段班的 AUC 約落在 **0.86 ~ 0.87**。只要先完成步驟 0、1、3、7，通常就能進入 0.85+。

## 步驟 0（最關鍵）：用機率而非類別標籤計算 AUC

這是目前分數極低的**主因**。

```python
# 原始程式（錯誤用法）
y_pred = grid_search.predict(X_test_scaled)   # 回傳 0 / 1 硬標籤
score = roc_auc_score(y_test, y_pred)         # AUC 退化
```

`roc_auc_score` 需要的是「分數／機率」來做排序，而不是硬切過的 0/1 類別。
傳入硬標籤會把模型的排序資訊全部丟掉，AUC 自然趨近 0.5。

**新增步驟：改用 `predict_proba` 的正類別機率（取第 1 欄＝違約機率）。**

```python
y_pred_proba = grid_search.predict_proba(X_test_scaled)[:, 1]
score = roc_auc_score(y_test, y_pred_proba)
```

> 影響：光是這一步，AUC 通常就會從 ~0.58 跳到 ~0.85 左右。

## 步驟 1：讓 GridSearchCV 直接以 AUC 為最佳化目標

`GridSearchCV` 對分類器**預設用 accuracy 評分**。在違約只佔約 6.7% 的不平衡資料下，
accuracy 會誤導參數選擇。

```python
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    scoring="roc_auc",   # ← 新增：直接優化 AUC
    n_jobs=4,
)
```

## 步驟 2：修正超參數搜尋範圍

原本的網格只在「極端值」之間搜尋，幾乎沒有調參效果：

```python
np.linspace(1, 500, 2, dtype=int)   # 只得到 [1, 500] 兩個值
```

`max_depth=1` 是樹樁、`n_estimators=1` 只有一棵樹，都是無意義的極端。

**新增步驟：改用合理且離散的候選值。**

```python
param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [6, 10, 16, None],
    "min_samples_split": [2, 10, 50],
    "min_samples_leaf": [1, 5, 20],
    "max_features": ["sqrt", "log2"],
}
```

> 若搜尋空間太大導致太慢，可改用 `RandomizedSearchCV(n_iter=30)` 取代 `GridSearchCV`。

## 步驟 3：處理缺失值（Imputation）

`MonthlyIncome` 缺約 30,000 筆、`NumberOfDependents` 缺約 4,000 筆。

**新增步驟：用中位數填補（對離群值較穩健），並可多加「是否缺失」旗標。**
`fit` 只能用訓練集，避免資料洩漏（data leakage）。

## 步驟 4：處理離群值與異常代碼

| 欄位 | 問題 | 建議處理 |
|------|------|----------|
| `RevolvingUtilizationOfUnsecuredLines` | 理論上 0~1，卻有 > 1 甚至上萬的值 | 上限截斷（cap） |
| `DebtRatio` | 有極端大值 | 用分位數截斷（如 99%） |
| 三個逾期欄位 | 出現 96、98 佔位碼 | 截斷為合理上限 |
| `age` | 出現 0 | 視為異常 |

## 步驟 5：處理類別不平衡

違約樣本只佔約 6.7%。讓模型知道類別權重，可改善排序能力。

```python
rf = RandomForestClassifier(random_state=0, class_weight="balanced")  # ← 新增
```

> 註：樹模型其實**不需要特徵縮放**，`StandardScaler` 對 AUC 沒有幫助（也不傷害）。

## 步驟 6：特徵工程

新增有意義的衍生特徵，通常能再擠出幾個百分點（每位成員收入、逾期總次數、月負債等）。

## 步驟 7：改用更強的模型（梯度提升）

在表格型信用資料上，梯度提升樹通常明顯優於 Random Forest，且 `HistGradientBoostingClassifier`
可**原生處理缺失值**。亦可改用 XGBoost / LightGBM（需 `pip install`）。

---
# 綜合範例：把上述步驟整合成可執行的改良流程

以下為一份**整合 0~9 步驟**的完整範例，可直接執行並輸出改良後的 AUC 與提交檔。

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df_train = pd.read_csv('./cs-training.csv')
df_test = pd.read_csv('./cs-test.csv')

late_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
]

def clean_features(df):
    """步驟 4 + 6：清理離群值/異常碼，並加入衍生特徵。"""
    X = df.iloc[:, 2:12].copy()
    # 步驟 4：逾期次數中的 96/98 佔位碼，截斷為合理上限
    for col in late_cols:
        X[col] = X[col].clip(upper=18)
    # 步驟 4：RevolvingUtilization / DebtRatio 用分位數截斷
    for col in ['RevolvingUtilizationOfUnsecuredLines', 'DebtRatio']:
        X[col] = X[col].clip(upper=X[col].quantile(0.99))
    # 步驟 6：特徵工程
    X['IncomePerPerson'] = X['MonthlyIncome'] / (X['NumberOfDependents'] + 1)
    X['TotalPastDue'] = X[late_cols].sum(axis=1)
    X['MonthlyDebt'] = X['DebtRatio'] * X['MonthlyIncome']
    X['HasBeenLate'] = (X['TotalPastDue'] > 0).astype(int)
    return X

X = clean_features(df_train)
y = df_train['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

In [2]:
# 步驟 7：HistGradientBoosting 可原生處理缺失值，免 imputer、免縮放
model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=500,
    l2_regularization=1.0,
    class_weight='balanced',   # 步驟 5：處理類別不平衡
    random_state=0,
)
model.fit(X_train, y_train)

# 步驟 0（最關鍵）：用 predict_proba 的正類別機率計算 AUC
y_pred_proba = model.predict_proba(X_test)[:, 1]
print('AUC-ROC:', roc_auc_score(y_test, y_pred_proba))

AUC-ROC: 0.8670032544999353


In [3]:
# 步驟 8：用交叉驗證的 AUC 評估，比單一切分更穩健
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=4)
print(f'CV AUC: {scores.mean():.4f} +/- {scores.std():.4f}')

CV AUC: 0.8655 +/- 0.0030


In [4]:
# 步驟 9：產生提交檔，Id 對齊官方欄位（cs-test.csv 的 Unnamed: 0 即正式 Id）
X_submit = clean_features(df_test)
submit_proba = model.predict_proba(X_submit)[:, 1]

submission = pd.DataFrame({
    'Id': df_test['Unnamed: 0'],
    'Probability': submit_proba,
})
submission.to_csv('submission_improved.csv', index=False)
submission.head()

,Id,Probability
0,1,0.459445
1,2,0.340090
2,3,0.268528
3,4,0.518088
4,5,0.653912
